# Proposition-Based Chunking for RAG

## A Deep-Dive Educational Notebook

This notebook provides a **complete, from-scratch implementation** of proposition-based chunking
for Retrieval-Augmented Generation (RAG). Every component — proposition extraction, embedding,
indexing, retrieval, and generation — is built with **native Python** (no LangChain, no LlamaIndex,
no OpenAI API).

| Component | Choice |
|---|---|
| **LLM** | `Qwen/Qwen3-14B` (4-bit NF4 quantization) |
| **Embeddings** | `BAAI/bge-base-en-v1.5` via sentence-transformers |
| **Vector Store** | FAISS (faiss-cpu) |
| **Tokenizer** | NLTK Punkt for sentence splitting |

> **Reference:** *Dense X Retrieval: What Retrieval Granularity Should We Use?*
> (Tong Chen, Hongwei Wang, Sihao Chen et al., 2023) — [arXiv:2312.06648](https://arxiv.org/abs/2312.06648)

## 1 · Environment Setup

Install all required packages. We pin `transformers≥4.51.0` for Qwen3 support.
We cache models on Google Drive so Colab sessions don't re-download every time.

In [ ]:
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy nltk

In [ ]:
import os, json, re, time, textwrap
import numpy as np
import torch
import faiss
import nltk

# ── Cache directory (Google Drive) ──────────────────────────
from google.colab import drive
drive.mount("/content/drive")

CACHE_DIR = "/content/drive/MyDrive/models"
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = CACHE_DIR

# ── NLTK sentence tokenizer ────────────────────────────────
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

print("✓ Imports complete")
print(f"  PyTorch  : {torch.__version__}")
print(f"  CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU      : {torch.cuda.get_device_name(0)}")
print(f"  Cache dir: {CACHE_DIR}")

### Load Qwen3-14B (4-bit NF4)

We use `BitsAndBytesConfig` for 4-bit NF4 quantization, which lets a 14B-parameter model
fit inside a single T4/A100 GPU with ~8 GB of VRAM.

After loading, we define a `generate()` helper — a thin wrapper that handles prompt
formatting, generation, and parsing. We use Qwen3's `enable_thinking=False` to disable
the internal reasoning block and get concise outputs.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-14B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=CACHE_DIR,
)
model.eval()
print(f"✓ Loaded {MODEL_ID}")
print(f"  dtype        : {next(model.parameters()).dtype}")
print(f"  device_map   : {model.hf_device_map}")

In [ ]:
def generate(prompt: str, max_new_tokens: int = 1024, temperature: float = 0.3) -> str:
    """Generate a completion from the Qwen3-14B model."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,        # Qwen3: disable <think> block
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=temperature > 0,
        )
    # Decode only the newly generated tokens
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# Quick smoke test
resp = generate("What is 2+2? Reply with just the number.")
print(f"Smoke test → {resp}")

In [ ]:
# ── Load Embedding Model ─────────────────────────────────────
# bge-base-en-v1.5: 110M-param bi-encoder, 768-dim, trained for retrieval
from sentence_transformers import SentenceTransformer

EMBED_MODEL_ID = "BAAI/bge-base-en-v1.5"
embed_model = SentenceTransformer(EMBED_MODEL_ID, cache_folder=CACHE_DIR)

# Quick test
test_vec = embed_model.encode(["hello world"])
print(f"✓ Loaded {EMBED_MODEL_ID}")
print(f"  Embedding dim: {test_vec.shape[1]}")
print(f"  Sample norm  : {np.linalg.norm(test_vec[0]):.4f}")

---
## 2 · What Are Propositions?

### Linguistic Background

In linguistics, a **proposition** is the smallest unit of meaning that can be either true or false.
It expresses a single, complete thought — one **subject–predicate relationship**.

| Concept | Example |
|---|---|
| **Sentence** | "Albert Einstein, who was born in Germany, developed the theory of relativity in 1905." |
| **Proposition 1** | Albert Einstein was born in Germany. |
| **Proposition 2** | Albert Einstein developed the theory of relativity. |
| **Proposition 3** | Albert Einstein developed the theory of relativity in 1905. |

A sentence can contain **multiple** propositions packed together through relative clauses,
conjunctions, appositives, and other syntactic devices.

### The Dense X Retrieval Paper (Chen et al., 2023)

The paper [*Dense X Retrieval: What Retrieval Granularity Should We Use?*](https://arxiv.org/abs/2312.06648)
(Chen, Wang, Chen et al., ICLR 2024) systematically studied three retrieval granularities:

1. **Document-level** — entire documents as retrieval units
2. **Passage-level** — fixed-size chunks (100–300 tokens)
3. **Proposition-level** — atomic factual statements

Key findings:
- Proposition-level retrieval achieved the **highest recall** across multiple QA benchmarks.
- It dramatically reduced retrieval noise — each retrieved unit contained exactly one fact.
- Downstream generation quality improved because the LLM received cleaner, more focused context.

The takeaway: **granularity matters**, and propositions sit at the sweet spot of specificity.

### Seeing Propositions in Action

Let's ask Qwen3 to decompose a single complex sentence into atomic propositions.

In [ ]:
example_sentence = (
    "Albert Einstein, who was born in Ulm, Germany in 1879, "
    "developed the special theory of relativity in 1905 while "
    "working at the Swiss Patent Office in Bern."
)

prompt = f"""Break the following sentence into a list of atomic propositions.

Rules:
1. Each proposition must express exactly ONE fact.
2. Each proposition must be self-contained (understandable without the original sentence).
3. Use full entity names, never pronouns.
4. Include dates and qualifiers where relevant.

Sentence: "{example_sentence}"

Return ONLY a JSON array of strings. No explanation."""

raw = generate(prompt, max_new_tokens=512, temperature=0.1)
print("Raw LLM output:")
print(raw)

# Parse the JSON array
try:
    propositions_demo = json.loads(raw)
except json.JSONDecodeError:
    # Fallback: extract the array with regex
    match = re.search(r'\[.*?\]', raw, re.DOTALL)
    propositions_demo = json.loads(match.group()) if match else raw.split("\n")

print(f"\n✓ Extracted {len(propositions_demo)} propositions:")
for i, p in enumerate(propositions_demo, 1):
    print(f"  {i}. {p}")

---
## 3 · Why Proposition Chunking?

### The Problem with Traditional Chunking

Traditional RAG systems split documents into fixed-size passages (e.g., 200 tokens with overlap).
This introduces two systematic problems:

| Problem | Description |
|---|---|
| **Fact Dilution** | A single chunk mixes multiple unrelated facts. When a query matches *one* fact, the chunk drags in irrelevant neighbours. |
| **Fact Fragmentation** | A fact that spans a chunk boundary gets split, so neither chunk contains the complete information. |

### How Propositions Fix This

Proposition-level chunking converts each document into **atomic factual statements** before indexing.
Each proposition:

- Contains **exactly one** subject–predicate relationship
- Is **self-contained** — understandable without surrounding context
- Uses **explicit entity names** — no pronouns or anaphora
- Includes **dates and qualifiers** where applicable

**Result:** every retrieved unit carries exactly one clean fact, dramatically reducing noise in
the LLM's context window.

In [ ]:
# Visual comparison: fixed-size chunk vs proposition-level
print("=" * 72)
print("FIXED-SIZE CHUNK (200 tokens)")
print("=" * 72)
print(textwrap.fill(
    "Marie Curie was born in Warsaw in 1867. She moved to Paris in 1891 to "
    "study at the Sorbonne. In 1903, she became the first woman to win a "
    "Nobel Prize, sharing the Physics prize with Pierre Curie and Henri "
    "Becquerel. She won a second Nobel Prize in Chemistry in 1911.",
    width=72,
))
print("\n→ Query: 'When did Marie Curie win the Chemistry Nobel?'")
print("→ The ENTIRE chunk is retrieved — 4 facts, only 1 is relevant.\n")

print("=" * 72)
print("PROPOSITION-LEVEL CHUNKS")
print("=" * 72)
props = [
    "Marie Curie was born in Warsaw in 1867.",
    "Marie Curie moved to Paris in 1891.",
    "Marie Curie studied at the Sorbonne.",
    "Marie Curie became the first woman to win a Nobel Prize in 1903.",
    "Marie Curie shared the 1903 Nobel Prize in Physics with Pierre Curie and Henri Becquerel.",
    "Marie Curie won the Nobel Prize in Chemistry in 1911.",
]
for i, p in enumerate(props, 1):
    print(f"  P{i}: {p}")
print("\n→ Query: 'When did Marie Curie win the Chemistry Nobel?'")
print("→ Only P6 is retrieved — exactly the fact needed. Zero noise.")

---
## 4 · Sentence vs. Proposition: A Detailed Comparison

A **sentence** is a grammatical unit; a **proposition** is a semantic unit.
One sentence can encode multiple propositions via:

- **Relative clauses** — "Einstein, *who was born in Germany*, developed …"
- **Conjunctions** — "Curie won *Physics* and later *Chemistry*"
- **Appositives** — "Turing, *a British mathematician*, broke …"
- **Temporal stacking** — "She studied at Sorbonne *and then* worked at the lab"

Let's measure this gap empirically on a multi-fact sentence.

In [ ]:
multi_fact_sentence = (
    "Marie Curie, a Polish-born physicist who moved to France, "
    "won the Nobel Prize in Physics in 1903 together with her husband "
    "Pierre Curie and Henri Becquerel for their research on radioactivity, "
    "and she later won a second Nobel Prize in Chemistry in 1911 for "
    "discovering polonium and radium."
)

# Count sentences
sentences = sent_tokenize(multi_fact_sentence)
print(f"Number of NLTK sentences: {len(sentences)}")
for i, s in enumerate(sentences, 1):
    print(f"  S{i}: {s}")

# Extract propositions
prompt = f"""Decompose the following sentence into atomic propositions.

Rules:
1. Each proposition = exactly ONE fact.
2. Self-contained (no pronouns, use full names).
3. Include dates and qualifiers.

Sentence: "{multi_fact_sentence}"

Return ONLY a JSON array of strings."""

raw = generate(prompt, max_new_tokens=512, temperature=0.1)
try:
    extracted = json.loads(raw)
except json.JSONDecodeError:
    match = re.search(r'\[.*?\]', raw, re.DOTALL)
    extracted = json.loads(match.group()) if match else []

print(f"\nNumber of PROPOSITIONS: {len(extracted)}")
for i, p in enumerate(extracted, 1):
    print(f"  P{i}: {p}")

print(f"\n→ Ratio: 1 sentence → {len(extracted)} propositions")
print("→ Sentence-level chunking would treat all these facts as one unit.")

---
## 5 · Extracting Propositions with an LLM

### Prompt Engineering Strategy

Our proposition-extraction prompt has four key design elements:

1. **Explicit rules** — numbered criteria the LLM must follow
2. **Few-shot example** — shows the input/output format concretely
3. **Structured output** — we ask for a JSON array for reliable parsing
4. **No-think mode** — we use Qwen3's `/no_think` to avoid verbose reasoning

Below we define a reusable `extract_propositions()` function.

In [ ]:
PROPOSITION_PROMPT_TEMPLATE = """You are a proposition extractor. Break the following text into
a list of atomic propositions.

Rules:
1. Each proposition must express exactly ONE fact or claim.
2. Each proposition must be self-contained and understandable without the original text.
3. Use full entity names — never use pronouns (he, she, it, they).
4. Include relevant dates, numbers, and qualifiers to make each fact precise.
5. Each proposition should have one subject-predicate relationship only.
6. Do NOT add information not present in the original text.

Example:
Input: "In 1969, Neil Armstrong became the first person to walk on the Moon during the Apollo 11 mission."
Output: ["Neil Armstrong walked on the Moon in 1969.", "Neil Armstrong was the first person to walk on the Moon.", "Neil Armstrong walked on the Moon during the Apollo 11 mission.", "The Apollo 11 mission took place in 1969."]

Text:
\"{text}\"

Return ONLY a valid JSON array of strings. No explanation, no markdown."""


def extract_propositions(text: str) -> list[str]:
    """Use Qwen3-14B to decompose text into atomic propositions."""
    prompt = PROPOSITION_PROMPT_TEMPLATE.format(text=text)
    raw = generate(prompt, max_new_tokens=1024, temperature=0.1)

    # Attempt JSON parse
    try:
        props = json.loads(raw)
        if isinstance(props, list):
            return [str(p).strip() for p in props if str(p).strip()]
    except json.JSONDecodeError:
        pass

    # Fallback: extract first JSON array from output
    match = re.search(r'\[.*?\]', raw, re.DOTALL)
    if match:
        try:
            props = json.loads(match.group())
            return [str(p).strip() for p in props if str(p).strip()]
        except json.JSONDecodeError:
            pass

    # Last resort: split on newlines, strip numbering
    lines = [re.sub(r'^\d+[\.\)]\s*', '', l).strip('" ') for l in raw.split("\n") if l.strip()]
    return [l for l in lines if len(l) > 10]

# Test on a paragraph
test_para = (
    "The Eiffel Tower, designed by Gustave Eiffel's engineering company, was built "
    "between 1887 and 1889 as the centerpiece of the 1889 World's Fair in Paris. "
    "Standing at 330 metres tall, it was the world's tallest structure until 1930 "
    "when the Chrysler Building in New York surpassed it."
)
props = extract_propositions(test_para)
print(f"Input paragraph ({len(test_para)} chars) → {len(props)} propositions:\n")
for i, p in enumerate(props, 1):
    print(f"  {i}. {p}")

---
## 6 · Building a Proposition-Level Index

### Synthetic Knowledge Base

We use a synthetic knowledge base of 8 paragraphs covering diverse scientific and historical
topics. This gives us rich multi-fact paragraphs that clearly demonstrate the benefits of
proposition-level decomposition.

In [ ]:
KNOWLEDGE_BASE = [
    {
        "id": "solar_system",
        "text": (
            "The Solar System formed approximately 4.6 billion years ago from the gravitational "
            "collapse of a giant molecular cloud. The Sun contains 99.86% of the system's total "
            "mass. Jupiter, the largest planet, has a mass of approximately 1.898 × 10^27 kg, "
            "which is more than twice the combined mass of all other planets. Saturn, known for "
            "its prominent ring system, has at least 146 confirmed moons, with Titan being the "
            "largest. Mars has two small moons named Phobos and Deimos."
        ),
    },
    {
        "id": "machine_learning",
        "text": (
            "Machine learning, a subset of artificial intelligence, was pioneered by Arthur "
            "Samuel in 1959 when he created a checkers-playing program at IBM. Deep learning, "
            "a subfield of machine learning, uses neural networks with multiple layers. The "
            "transformer architecture, introduced by Vaswani et al. in the 2017 paper "
            "'Attention Is All You Need', revolutionized natural language processing. GPT-3, "
            "released by OpenAI in June 2020, contains 175 billion parameters."
        ),
    },
    {
        "id": "human_body",
        "text": (
            "The adult human body contains approximately 37.2 trillion cells. The human heart "
            "beats about 100,000 times per day, pumping roughly 7,570 litres of blood. The "
            "small intestine is approximately 6 metres long and is the primary site of nutrient "
            "absorption. The human brain, weighing about 1.4 kilograms, contains roughly 86 "
            "billion neurons connected by approximately 100 trillion synapses."
        ),
    },
    {
        "id": "world_war_2",
        "text": (
            "World War II began on September 1, 1939 when Nazi Germany invaded Poland. The "
            "conflict eventually involved more than 30 countries and resulted in an estimated "
            "70-85 million fatalities. The D-Day invasion on June 6, 1944 saw Allied forces "
            "land on the beaches of Normandy, France, marking a turning point in the European "
            "theatre. The war in Europe ended on May 8, 1945, known as V-E Day, when Germany "
            "surrendered unconditionally."
        ),
    },
    {
        "id": "climate_science",
        "text": (
            "Carbon dioxide concentrations in Earth's atmosphere reached 421 parts per million "
            "in 2023, the highest level in at least 800,000 years. The global average temperature "
            "has risen by approximately 1.1°C since the pre-industrial era. The Paris Agreement, "
            "adopted in 2015, aims to limit global warming to 1.5°C above pre-industrial levels. "
            "The Intergovernmental Panel on Climate Change (IPCC) was established in 1988 by the "
            "United Nations Environment Programme and the World Meteorological Organization."
        ),
    },
    {
        "id": "ancient_egypt",
        "text": (
            "The Great Pyramid of Giza was built around 2560 BCE as a tomb for Pharaoh Khufu. "
            "It originally stood 146.6 metres tall and was the tallest man-made structure for "
            "over 3,800 years. The ancient Egyptians developed hieroglyphic writing around 3200 "
            "BCE. The Rosetta Stone, discovered in 1799 by French soldiers, enabled Jean-François "
            "Champollion to decipher hieroglyphs in 1822. Cleopatra VII, the last active ruler "
            "of the Ptolemaic Kingdom of Egypt, died in 30 BCE."
        ),
    },
    {
        "id": "quantum_physics",
        "text": (
            "Quantum mechanics emerged in the early 20th century through contributions from Max "
            "Planck, Albert Einstein, Niels Bohr, and Werner Heisenberg. Heisenberg's uncertainty "
            "principle, formulated in 1927, states that certain pairs of physical properties cannot "
            "both be known to arbitrary precision simultaneously. Schrödinger's cat, proposed by "
            "Erwin Schrödinger in 1935, is a thought experiment illustrating quantum superposition. "
            "Quantum entanglement, which Einstein called 'spooky action at a distance', was "
            "experimentally confirmed by Alain Aspect in 1982."
        ),
    },
    {
        "id": "internet_history",
        "text": (
            "ARPANET, the precursor to the modern Internet, was established in 1969 by the U.S. "
            "Department of Defense. Tim Berners-Lee invented the World Wide Web in 1989 while "
            "working at CERN in Switzerland. The first web browser, called WorldWideWeb, was "
            "created by Berners-Lee in 1990. Google was founded by Larry Page and Sergey Brin "
            "in September 1998 while they were PhD students at Stanford University. By 2023, "
            "approximately 5.3 billion people worldwide were using the Internet."
        ),
    },
]

print(f"✓ Knowledge base: {len(KNOWLEDGE_BASE)} paragraphs")
total_chars = sum(len(d['text']) for d in KNOWLEDGE_BASE)
print(f"  Total characters: {total_chars:,}")
for doc in KNOWLEDGE_BASE:
    print(f"  • {doc['id']:20s} — {len(doc['text']):4d} chars")

### Step 1: Sentence-Level Baseline

First, let's create a **sentence-level** index as our baseline. We use NLTK's Punkt tokenizer
to split each paragraph into individual sentences.

In [ ]:
sentence_chunks = []

for doc in KNOWLEDGE_BASE:
    sents = sent_tokenize(doc["text"])
    for s in sents:
        sentence_chunks.append({"text": s.strip(), "source": doc["id"]})

print(f"✓ Sentence-level chunking: {len(sentence_chunks)} chunks")
print(f"\nSample sentences from 'solar_system':")
for ch in sentence_chunks:
    if ch["source"] == "solar_system":
        print(f"  • {ch['text'][:90]}...")

### Step 2: Proposition Extraction

Now we run every paragraph through our LLM-based proposition extractor.
This is the core step — each paragraph is decomposed into atomic facts.

In [ ]:
proposition_chunks = []

print("Extracting propositions from knowledge base...\n")
for doc in KNOWLEDGE_BASE:
    t0 = time.time()
    props = extract_propositions(doc["text"])
    elapsed = time.time() - t0
    print(f"  {doc['id']:20s} → {len(props):2d} propositions  ({elapsed:.1f}s)")
    for p in props:
        proposition_chunks.append({"text": p, "source": doc["id"]})

print(f"\n✓ Total propositions: {len(proposition_chunks)}")
print(f"  vs. sentences:      {len(sentence_chunks)}")
print(f"  Expansion ratio:    {len(proposition_chunks)/len(sentence_chunks):.2f}x")

### Step 3: Inspect Proposition Quality

Let's compare the original sentences to the extracted propositions for one paragraph.

In [ ]:
inspect_id = "ancient_egypt"
doc_text = next(d["text"] for d in KNOWLEDGE_BASE if d["id"] == inspect_id)

print(f"ORIGINAL PARAGRAPH ({inspect_id}):")
print(textwrap.fill(doc_text, width=80))

print(f"\nSENTENCES ({len([c for c in sentence_chunks if c['source'] == inspect_id])}):")
for i, ch in enumerate([c for c in sentence_chunks if c["source"] == inspect_id], 1):
    print(f"  S{i}: {ch['text']}")

print(f"\nPROPOSITIONS ({len([c for c in proposition_chunks if c['source'] == inspect_id])}):")
for i, ch in enumerate([c for c in proposition_chunks if c["source"] == inspect_id], 1):
    print(f"  P{i}: {ch['text']}")

### Step 4: Embed and Build FAISS Indexes

We create **two** FAISS indexes:
1. **Sentence-level index** — baseline
2. **Proposition-level index** — our proposition chunking approach

Both use the same embedding model (`bge-base-en-v1.5`) for fair comparison.

In [ ]:
def build_faiss_index(chunks: list[dict]) -> tuple:
    """Embed texts and build a FAISS L2 index. Returns (index, texts, sources)."""
    texts = [ch["text"] for ch in chunks]
    sources = [ch["source"] for ch in chunks]

    # bge-base-en-v1.5 recommends prepending "Represent this sentence: " for documents
    embeddings = embed_model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
    embeddings = np.array(embeddings, dtype="float32")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product = cosine similarity when normalized
    index.add(embeddings)

    return index, texts, sources

print("Building sentence-level index...")
sent_index, sent_texts, sent_sources = build_faiss_index(sentence_chunks)
print(f"  → {sent_index.ntotal} vectors, dim={sent_index.d}")

print("\nBuilding proposition-level index...")
prop_index, prop_texts, prop_sources = build_faiss_index(proposition_chunks)
print(f"  → {prop_index.ntotal} vectors, dim={prop_index.d}")

In [ ]:
# ── Search helper ────────────────────────────────────────────
def search(query: str, index, texts, sources, k: int = 5) -> list[dict]:
    """Search a FAISS index and return top-k results."""
    # bge recommends "Represent this sentence: " prefix for queries too
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, k)
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
        if idx < 0:
            continue
        results.append({
            "rank": rank + 1,
            "text": texts[idx],
            "source": sources[idx],
            "score": float(score),
        })
    return results

# Quick test
test_results = search("How many moons does Saturn have?", prop_index, prop_texts, prop_sources, k=3)
print("Quick search test — 'How many moons does Saturn have?'")
for r in test_results:
    print(f"  #{r['rank']} [{r['source']}] (score={r['score']:.4f})")
    print(f"     {r['text']}")

---
## 7 · Retrieval Comparison: Sentences vs. Propositions

Now for the key experiment. We run the **same queries** against both indexes and compare
what gets retrieved. The hypothesis: proposition-level retrieval returns more focused,
directly relevant results with less noise.

In [ ]:
TEST_QUERIES = [
    "When was the Great Pyramid of Giza built?",
    "How many parameters does GPT-3 have?",
    "What is Heisenberg's uncertainty principle?",
    "Who invented the World Wide Web?",
    "How many neurons are in the human brain?",
    "When did World War II end in Europe?",
    "What is the goal of the Paris Agreement?",
    "How many moons does Saturn have?",
]

print(f"Running {len(TEST_QUERIES)} test queries against both indexes...\n")
print(f"{'Query':<48} {'Sent. Top-1 (score)':<42} {'Prop. Top-1 (score)'}")
print("─" * 132)

for query in TEST_QUERIES:
    s_res = search(query, sent_index, sent_texts, sent_sources, k=1)
    p_res = search(query, prop_index, prop_texts, prop_sources, k=1)

    s_top = f"{s_res[0]['text'][:35]}… ({s_res[0]['score']:.3f})" if s_res else "—"
    p_top = f"{p_res[0]['text'][:35]}… ({p_res[0]['score']:.3f})" if p_res else "—"

    print(f"{query:<48} {s_top:<42} {p_top}")

### Deep Dive: Side-by-Side Retrieval

Let's look at full top-5 results for a few queries to really see the difference.

In [ ]:
def compare_retrieval(query: str, k: int = 5):
    """Print side-by-side comparison for a query."""
    s_results = search(query, sent_index, sent_texts, sent_sources, k=k)
    p_results = search(query, prop_index, prop_texts, prop_sources, k=k)

    print(f"\n{'='*80}")
    print(f"QUERY: {query}")
    print(f"{'='*80}")

    print(f"\n  SENTENCE-LEVEL RETRIEVAL (top {k}):")
    for r in s_results:
        print(f"    #{r['rank']} [{r['source']}] score={r['score']:.4f}")
        print(f"       {r['text']}")

    print(f"\n  PROPOSITION-LEVEL RETRIEVAL (top {k}):")
    for r in p_results:
        print(f"    #{r['rank']} [{r['source']}] score={r['score']:.4f}")
        print(f"       {r['text']}")

# Detailed comparison for 3 representative queries
compare_retrieval("Who deciphered Egyptian hieroglyphs?", k=5)
compare_retrieval("When was the transformer architecture introduced?", k=5)
compare_retrieval("How much blood does the human heart pump per day?", k=5)

### Precision Analysis

We measure **top-1 relevance**: does the top result directly and precisely answer the query?
We use the LLM as a judge to score relevance on a 1–5 scale.

In [ ]:
def judge_relevance(query: str, retrieved_text: str) -> int:
    """Use the LLM to judge relevance of a retrieved passage to a query (1-5)."""
    prompt = f"""Rate how well the following retrieved text answers the query.
Score from 1 to 5:
  1 = completely irrelevant
  2 = vaguely related but doesn't answer
  3 = partially answers
  4 = mostly answers
  5 = directly and precisely answers

Query: "{query}"
Retrieved text: "{retrieved_text}"

Return ONLY a single integer (1-5). Nothing else."""
    raw = generate(prompt, max_new_tokens=16, temperature=0.0)
    match = re.search(r'[1-5]', raw)
    return int(match.group()) if match else 3

print("Judging top-1 relevance for each query...\n")
print(f"{'Query':<50} {'Sent.':<8} {'Prop.':<8} {'Winner'}")
print("─" * 80)

sent_scores = []
prop_scores = []

for query in TEST_QUERIES:
    s_res = search(query, sent_index, sent_texts, sent_sources, k=1)
    p_res = search(query, prop_index, prop_texts, prop_sources, k=1)

    s_score = judge_relevance(query, s_res[0]["text"]) if s_res else 0
    p_score = judge_relevance(query, p_res[0]["text"]) if p_res else 0

    sent_scores.append(s_score)
    prop_scores.append(p_score)

    winner = "Prop ✓" if p_score > s_score else ("Sent ✓" if s_score > p_score else "Tie")
    print(f"{query:<50} {s_score:<8} {p_score:<8} {winner}")

print(f"\n{'─'*80}")
print(f"{'AVERAGE':<50} {np.mean(sent_scores):<8.2f} {np.mean(prop_scores):<8.2f}")
print(f"\nProposition-level wins: {sum(p > s for p, s in zip(prop_scores, sent_scores))}/{len(TEST_QUERIES)}")
print(f"Sentence-level wins:   {sum(s > p for p, s in zip(prop_scores, sent_scores))}/{len(TEST_QUERIES)}")
print(f"Ties:                  {sum(p == s for p, s in zip(prop_scores, sent_scores))}/{len(TEST_QUERIES)}")

### Token Efficiency: Less Noise per Retrieval

An often overlooked advantage: proposition-level retrieval puts **fewer tokens** into the
LLM's context window per retrieved unit, leaving more room for the question and generation.

In [ ]:
sent_lengths = [len(t) for t in sent_texts]
prop_lengths = [len(t) for t in prop_texts]

print("CHARACTER LENGTH STATISTICS")
print(f"{'Metric':<25} {'Sentences':>12} {'Propositions':>14}")
print("─" * 55)
print(f"{'Count':<25} {len(sent_lengths):>12} {len(prop_lengths):>14}")
print(f"{'Mean length (chars)':<25} {np.mean(sent_lengths):>12.1f} {np.mean(prop_lengths):>14.1f}")
print(f"{'Median length (chars)':<25} {np.median(sent_lengths):>12.1f} {np.median(prop_lengths):>14.1f}")
print(f"{'Max length (chars)':<25} {max(sent_lengths):>12} {max(prop_lengths):>14}")
print(f"{'Min length (chars)':<25} {min(sent_lengths):>12} {min(prop_lengths):>14}")

# Context window usage for top-5 retrieval
for query in TEST_QUERIES[:3]:
    s_res = search(query, sent_index, sent_texts, sent_sources, k=5)
    p_res = search(query, prop_index, prop_texts, prop_sources, k=5)
    s_chars = sum(len(r["text"]) for r in s_res)
    p_chars = sum(len(r["text"]) for r in p_res)
    print(f"\nQuery: '{query[:45]}...'")
    print(f"  Sentence top-5 total chars: {s_chars:>5}")
    print(f"  Proposition top-5 total chars: {p_chars:>5} ({p_chars/s_chars*100:.0f}% of sentence)")

---
## 8 · Complete RAG Pipeline with Proposition Chunking

Now we tie everything together into a full end-to-end RAG pipeline:

1. **Retrieve** — find top-k propositions matching the query
2. **Augment** — format retrieved propositions into a context block
3. **Generate** — ask the LLM to answer using only the retrieved context

This is proposition-based RAG, implemented entirely from scratch.

In [ ]:
def rag_answer(query: str, k: int = 5, use_propositions: bool = True) -> dict:
    """Full RAG pipeline: retrieve → augment → generate."""
    # 1. RETRIEVE
    if use_propositions:
        results = search(query, prop_index, prop_texts, prop_sources, k=k)
        method = "proposition"
    else:
        results = search(query, sent_index, sent_texts, sent_sources, k=k)
        method = "sentence"

    # 2. AUGMENT — build context
    context_parts = []
    for r in results:
        context_parts.append(f"[{r['source']}] {r['text']}")
    context = "\n".join(context_parts)

    # 3. GENERATE
    rag_prompt = f"""Answer the following question using ONLY the provided context.
If the context doesn't contain enough information, say "I don't have enough information."

Context:
{context}

Question: {query}

Provide a concise, factual answer."""

    answer = generate(rag_prompt, max_new_tokens=256, temperature=0.1)

    return {
        "query": query,
        "method": method,
        "answer": answer,
        "retrieved": results,
        "context_chars": len(context),
    }

print("✓ RAG pipeline defined")

### RAG Pipeline Demo

Let's run several queries through the full pipeline and compare proposition-based vs
sentence-based RAG answers.

In [ ]:
RAG_QUERIES = [
    "When and by whom was the World Wide Web invented?",
    "What is the relationship between Heisenberg's uncertainty principle and quantum mechanics?",
    "How tall was the Great Pyramid of Giza originally?",
    "When did World War II begin and when did it end in Europe?",
    "What is the Paris Agreement's temperature goal?",
]

for query in RAG_QUERIES:
    print(f"\n{'='*80}")
    print(f"Q: {query}")
    print(f"{'='*80}")

    # Proposition-based RAG
    prop_result = rag_answer(query, k=5, use_propositions=True)
    print(f"\n  [PROPOSITION RAG] ({prop_result['context_chars']} context chars)")
    print(f"  Retrieved:")
    for r in prop_result["retrieved"][:3]:
        print(f"    • [{r['source']}] {r['text']}")
    print(f"  Answer: {prop_result['answer']}")

    # Sentence-based RAG
    sent_result = rag_answer(query, k=5, use_propositions=False)
    print(f"\n  [SENTENCE RAG] ({sent_result['context_chars']} context chars)")
    print(f"  Retrieved:")
    for r in sent_result["retrieved"][:3]:
        print(f"    • [{r['source']}] {r['text']}")
    print(f"  Answer: {sent_result['answer']}")

### RAG Answer Quality Assessment

Let's use the LLM to grade the quality of answers from both approaches.

In [ ]:
def judge_answer_quality(query: str, answer: str, method: str) -> dict:
    """Judge RAG answer quality on faithfulness and completeness."""
    prompt = f"""Rate the following answer to a question.

Question: "{query}"
Answer: "{answer}"

Score on two dimensions (1-5 each):
1. FAITHFULNESS: Does the answer only state facts? (5 = perfectly factual)
2. COMPLETENESS: Does it fully address the question? (5 = fully complete)

Return ONLY valid JSON: {{"faithfulness": N, "completeness": N}}"""

    raw = generate(prompt, max_new_tokens=64, temperature=0.0)
    try:
        scores = json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        scores = json.loads(match.group()) if match else {"faithfulness": 3, "completeness": 3}
    scores["method"] = method
    return scores

print(f"{'Query':<55} {'Method':<12} {'Faith.':<8} {'Compl.':<8}")
print("─" * 85)

all_scores = {"prop": [], "sent": []}

for query in RAG_QUERIES:
    prop_result = rag_answer(query, k=5, use_propositions=True)
    sent_result = rag_answer(query, k=5, use_propositions=False)

    p_scores = judge_answer_quality(query, prop_result["answer"], "prop")
    s_scores = judge_answer_quality(query, sent_result["answer"], "sent")

    all_scores["prop"].append(p_scores)
    all_scores["sent"].append(s_scores)

    print(f"{query:<55} {'Prop':<12} {p_scores.get('faithfulness','-'):<8} {p_scores.get('completeness','-'):<8}")
    print(f"{'':<55} {'Sent':<12} {s_scores.get('faithfulness','-'):<8} {s_scores.get('completeness','-'):<8}")

# Averages
print(f"\n{'─'*85}")
p_faith = np.mean([s.get("faithfulness", 3) for s in all_scores["prop"]])
p_comp  = np.mean([s.get("completeness", 3) for s in all_scores["prop"]])
s_faith = np.mean([s.get("faithfulness", 3) for s in all_scores["sent"]])
s_comp  = np.mean([s.get("completeness", 3) for s in all_scores["sent"]])
print(f"{'AVERAGE':<55} {'Prop':<12} {p_faith:<8.2f} {p_comp:<8.2f}")
print(f"{'':<55} {'Sent':<12} {s_faith:<8.2f} {s_comp:<8.2f}")

---
## 9 · Tradeoffs and When to Use Proposition Chunking

### Quantitative Summary

In [ ]:
print("PROPOSITION CHUNKING — COST / BENEFIT ANALYSIS")
print("=" * 65)
print(f"\n  Knowledge base paragraphs:   {len(KNOWLEDGE_BASE)}")
print(f"  Sentence-level chunks:       {len(sentence_chunks)}")
print(f"  Proposition-level chunks:    {len(proposition_chunks)}")
print(f"  Chunk expansion factor:      {len(proposition_chunks)/len(sentence_chunks):.2f}x")
print(f"\n  Avg sentence length:         {np.mean(sent_lengths):.0f} chars")
print(f"  Avg proposition length:      {np.mean(prop_lengths):.0f} chars")
print(f"  Length ratio:                {np.mean(prop_lengths)/np.mean(sent_lengths):.2f}x")

# Storage comparison
sent_storage = sent_index.ntotal * sent_index.d * 4  # float32
prop_storage = prop_index.ntotal * prop_index.d * 4
print(f"\n  Sentence index storage:      {sent_storage/1024:.1f} KB ({sent_index.ntotal} vectors)")
print(f"  Proposition index storage:   {prop_storage/1024:.1f} KB ({prop_index.ntotal} vectors)")
print(f"  Storage increase:            {prop_storage/sent_storage:.2f}x")

print(f"\n  Avg top-1 relevance (sent):  {np.mean(sent_scores):.2f}/5")
print(f"  Avg top-1 relevance (prop):  {np.mean(prop_scores):.2f}/5")

### When to Use Proposition Chunking

| Scenario | Recommendation | Why |
|---|---|---|
| **Fact-heavy documents** (scientific, legal, financial) | ✅ **Use propositions** | Dense multi-fact sentences benefit most from decomposition |
| **Simple factoid QA** | ✅ **Use propositions** | Queries target single facts — propositions align perfectly |
| **Narrative / opinion text** | ⚠️ **Consider carefully** | Meaning may depend on surrounding context |
| **Long-form generation** | ⚠️ **Hybrid approach** | Retrieve propositions, but also pull full paragraphs for context |
| **Latency-critical systems** | ⚠️ **Profile first** | More chunks = more embedding + search cost |
| **Small knowledge base** | ❌ **Likely overkill** | Sentence-level may suffice when there's little retrieval noise |

### Key Tradeoffs

**Costs of Proposition Chunking:**
- 🔴 **LLM preprocessing cost** — every paragraph needs an LLM call for decomposition
- 🔴 **More vectors to store** — typically 2-4× more chunks than sentence-level
- 🔴 **Higher indexing latency** — more embeddings to compute
- 🔴 **Risk of LLM extraction errors** — hallucinated or lost facts during decomposition

**Benefits of Proposition Chunking:**
- 🟢 **Dramatically reduced retrieval noise** — each chunk = one fact
- 🟢 **Higher top-k precision** — less dilution in retrieved context
- 🟢 **More efficient context usage** — shorter chunks leave room for more retrieval or generation
- 🟢 **Better for specific queries** — exact matches on atomic facts
- 🟢 **Cleaner RAG answers** — LLM receives focused context, generates better responses

### Advanced: Hybrid Approach (Proposition + Parent Chunk)

In production, many teams use a **hybrid** strategy:
1. Index propositions for fine-grained retrieval
2. But return the **parent paragraph** for richer context

This gets the best of both worlds: precise retrieval + contextual generation.

In [ ]:
# Build a parent lookup: proposition → original paragraph
prop_to_parent = {}
for ch in proposition_chunks:
    parent = next(d["text"] for d in KNOWLEDGE_BASE if d["id"] == ch["source"])
    prop_to_parent[ch["text"]] = parent

def hybrid_rag(query: str, k: int = 5) -> dict:
    """Retrieve propositions but return parent paragraphs for context."""
    results = search(query, prop_index, prop_texts, prop_sources, k=k)

    # Deduplicate parents (multiple propositions may share the same parent)
    seen_sources = set()
    parent_context_parts = []
    for r in results:
        if r["source"] not in seen_sources:
            seen_sources.add(r["source"])
            parent = prop_to_parent[r["text"]]
            parent_context_parts.append(f"[{r['source']}] {parent}")

    context = "\n\n".join(parent_context_parts)

    rag_prompt = f"""Answer the following question using ONLY the provided context.
If the context doesn't contain enough information, say "I don't have enough information."

Context:
{context}

Question: {query}

Provide a concise, factual answer."""

    answer = generate(rag_prompt, max_new_tokens=256, temperature=0.1)

    return {
        "query": query,
        "answer": answer,
        "propositions_used": [r["text"] for r in results],
        "parents_used": list(seen_sources),
        "context_chars": len(context),
    }

# Demo
hybrid_result = hybrid_rag("What were the key events at the end of World War II in Europe?")
print(f"Query: {hybrid_result['query']}")
print(f"\nPropositions retrieved:")
for p in hybrid_result["propositions_used"]:
    print(f"  • {p}")
print(f"\nParent paragraphs used: {hybrid_result['parents_used']}")
print(f"Context size: {hybrid_result['context_chars']} chars")
print(f"\nAnswer: {hybrid_result['answer']}")

---
## 10 · Summary

### What We Built

In this notebook, we implemented **proposition-based chunking for RAG** entirely from scratch:

1. **Proposition Extractor** — LLM-based decomposition of paragraphs into atomic facts
2. **Dual FAISS Indexes** — sentence-level (baseline) and proposition-level
3. **Retrieval Comparison** — empirical evidence that propositions yield more focused results
4. **Complete RAG Pipeline** — end-to-end question answering with both approaches
5. **Quality Assessment** — LLM-as-judge evaluation of retrieval relevance and answer quality
6. **Hybrid Approach** — proposition retrieval with parent-paragraph context

### Key Takeaways

- **Propositions are the optimal retrieval granularity** for fact-heavy knowledge bases
- **One sentence ≠ one fact** — sentences routinely pack 2-5 propositions
- **Precision matters more than recall** in RAG — retrieving exactly what's needed beats retrieving everything
- **The cost is real** — more preprocessing, more storage, more embeddings
- **Hybrid approaches** offer the best of both worlds in production systems

### References

1. Chen, T., Wang, H., Chen, S. et al. (2023). *Dense X Retrieval: What Retrieval Granularity Should We Use?*
   [arXiv:2312.06648](https://arxiv.org/abs/2312.06648)
2. Reimers, N. & Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks.*
   [arXiv:1908.10084](https://arxiv.org/abs/1908.10084)
3. Johnson, J., Douze, M. & Jégou, H. (2019). *Billion-scale similarity search with GPUs.*
   IEEE Transactions on Big Data.